# 57 같은 카드의 공통 조건 원문 보완

56의 DB/추출 선택이 동결된 뒤 실행한다. 원질문 → 구조화 조회 → 조회 카드 안에서 vector+BM25 → 기존 프로필 재정렬까지가 A다. 구조화 결과0이면 원래 상품/발급사 범위에서 비구조화 검색을 한 번 수행한다.

B는 A의 카드·순위·근거를 모두 유지하고 같은 카드 안에서 `원질문 + 전월실적 한도 횟수 서비스 선택 제외 조건`으로 FTS/BM25를 조회한다. 제목/경로의 고정 조건 키워드로 걸러 기존 BGE로 정렬하고, 동점 chunk ID 사전순으로 최대2개 원문 구역을 통째로 추가한다. A최대5+B최대2=카드별 최대7, 전체64000bytes. parent bundle 하나의 원문청크 최대5는 바꾸지 않는다.

신규 임베딩/API0. 보완 근거 수가 증가하는 실험이며 동일 입력량 실험이 아니다. 원문 정답·gold fact ID·정답 페이지는 선택 함수에 제공하지 않는다. `.py`는 58과 공유할 추가 규칙 검증과 장시간 CPU 검색 재개만 소유한다. HTML은 결과를 확인하고 직접 작성한다.

In [1]:
from pathlib import Path
import importlib.util,sys,json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'notebooks/scripts').is_dir())
spec=importlib.util.spec_from_file_location('runtime57_notebook',ROOT/'notebooks/scripts/57_sequential_evidence_runtime.py')
T=importlib.util.module_from_spec(spec);sys.modules[spec.name]=T;spec.loader.exec_module(T)
queries,plans,manifest=T.prepare(ROOT/'notebooks/57_sequential_condition_evidence.ipynb')
print('동결 질문',len(queries),'프로필',len(T.X.PROFILES))

notebook57 cell 1
동결 질문 40 프로필 2


In [2]:
resources=T.run(queries,plans,manifest)
print(json.dumps(resources,ensure_ascii=False,indent=2))

notebook57 cell 2
57 retrieval 1 /80 H01:card_page_section_benefit
`torch_dtype` is deprecated! Use `dtype` instead!
57 retrieval 2 /80 H01:parent_child_bundle
57 retrieval 3 /80 H02:card_page_section_benefit
57 retrieval 4 /80 H02:parent_child_bundle
57 retrieval 5 /80 H03:card_page_section_benefit
57 retrieval 6 /80 H03:parent_child_bundle
57 retrieval 7 /80 H04:card_page_section_benefit
57 retrieval 8 /80 H04:parent_child_bundle
57 retrieval 9 /80 H05:card_page_section_benefit
57 retrieval 10 /80 H05:parent_child_bundle
57 retrieval 11 /80 H06:card_page_section_benefit
57 retrieval 12 /80 H06:parent_child_bundle
57 retrieval 13 /80 H07:card_page_section_benefit
57 retrieval 14 /80 H07:parent_child_bundle
57 retrieval 15 /80 H08:card_page_section_benefit
57 retrieval 16 /80 H08:parent_child_bundle
57 retrieval 17 /80 H09:card_page_section_benefit
57 retrieval 18 /80 H09:parent_child_bundle
57 retrieval 19 /80 H10:card_page_section_benefit
57 retrieval 20 /80 H10:parent_child_bundle
5

## 원문 사실 판정 후 선택

검색 완료 자체를 개선으로 판단하지 않는다. 같은 고정 질문·원문 사실 단위로 A/B의 필요한 근거 회수, 완전근거 확보, 카드 범위, 다른 혜택 조건 연결/충돌을 수동 검토한다. 완전근거 개선과 중요 관계 오류0이 함께 확인되지 않으면 A를 유지한다. source fact 판정과 56 추출 조건 정답은 검색 함수의 입력에 들어가지 않는다. 이후 58의 생성 비교는 별도 단계다.

## 실행·검토 결과
검색80건을 실행했다. 37질문의 필수 근거 회수율은 benefit A/B 모두97.34%, 완전근거30/37이다. parent-child는 A94.32%·28/37에서 B96.89%·31/37로 개선했다. H24/D36/D38의 완전근거가 채워졌고 D40은 실적만 보완돼 앱 범위의 모호함이 남았다.
독립 검토에서 D39/D40 교통 실적의 원문 범위 증명을 생활서비스4쪽에서 교통특화2쪽으로 바로잡았다. 별도 source_facts.json에서 증명만 수정하고55 원본·ID·값·분모는 보존했다. 재계산 수치는 동일했다. H11 parent A/B의40만원은 이번에 실제 본문을 다시 확인한 판정이므로55와의 단순 개선량 비교를 하지 않는다.
교정 후 독립 검토 완료: benefit A만, parent-child A/B를58 최종 선택 후보로 허용한다. 추가 조립은 원색인 청크의 텍스트·카드·제목·페이지를 그대로 보존했다. 기존 OCR과 제목 관계 전체를 검증한 것은 아니다. API/새임베딩0, 검색 실행348.93초; 생성시간은 아직 미측정이다. 전체 판정·검토·자원·후속 허용 계약은 data/57_sequential_condition_evidence/와 같은 번호 HTML에 있다.
